# 🦀 Day 1: Feature 1 — TCP Server and Client
---

Welcome to **Week 22: Build Phase**! Today we add networking to RustKV.

- **TCP server**: Build the server using tokio, accept connections, spawn per-client tasks
- **Command handling**: Read commands from TcpStream, parse, execute, respond
- **Client implementation**: Connect, send commands, read responses
- **Connection handling**: Graceful disconnect, error recovery
- **Exercise**: Add connection timeout and max-connections limit

## Add tokio dependency

In a Cargo project, add to `Cargo.toml`:

```toml
[dependencies]
tokio = { version = "1", features = ["full"] }
```

For EvCxR notebook, use `:dep` to add it inline.

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

// Server main loop structure
use tokio::net::{TcpListener, TcpStream};
use tokio::io::{AsyncReadExt, AsyncWriteExt};
use std::sync::Arc;

async fn handle_client(mut stream: TcpStream, storage: Arc<std::sync::RwLock<std::collections::HashMap<String, String>>>) {
    let mut buf = vec![0u8; 1024];
    loop {
        match stream.read(&mut buf).await {
            Ok(0) => break,  // Connection closed
            Ok(n) => {
                let cmd = String::from_utf8_lossy(&buf[..n]);
                let response = execute_command(&cmd, &storage);
                if stream.write_all(response.as_bytes()).await.is_err() { break; }
            }
            Err(_) => break,
        }
    }
}

fn execute_command(cmd: &str, storage: &std::sync::RwLock<std::collections::HashMap<String, String>>) -> String {
    let parts: Vec<&str> = cmd.trim().split_whitespace().collect();
    match parts.as_slice() {
        ["GET", key] => {
            let s = storage.read().unwrap();
            s.get(*key).cloned().unwrap_or_else(|| "(nil)".to_string())
        }
        ["SET", key, value] => {
            let mut s = storage.write().unwrap();
            s.insert(key.to_string(), value.to_string());
            "OK".to_string()
        }
        ["PING"] => "PONG".to_string(),
        _ => "ERR unknown command".to_string(),
    }
}

println!("Server handler and execute_command defined.");

In [ ]:
// Simulated server startup (no actual bind in notebook — use in full project)

let server_code = r#"
async fn run_server(port: u16) -> Result<(), Box<dyn std::error::Error + Send + Sync>> {
    let listener = TcpListener::bind(format!("127.0.0.1:{}", port)).await?;
    let storage = Arc::new(RwLock::new(HashMap::new()));
    loop {
        let (stream, _) = listener.accept().await?;
        let storage = Arc::clone(&storage);
        tokio::spawn(async move { handle_client(stream, storage).await; });
    }
}

// In main: tokio::runtime::Runtime::new()?.block_on(run_server(6379))
"#;

println!("Server main loop:");
println!("{}", server_code);

In [ ]:
// Client implementation — connect, send, read

async fn client_send_command(host: &str, port: u16, cmd: &str) -> Result<String, Box<dyn std::error::Error + Send + Sync>> {
    let mut stream = TcpStream::connect(format!("{}:{}", host, port)).await?;
    stream.write_all(cmd.as_bytes()).await?;
    stream.write_all(b"\n").await?;
    let mut buf = vec![0u8; 1024];
    let n = stream.read(&mut buf).await?;
    Ok(String::from_utf8_lossy(&buf[..n]).trim().to_string())
}

// Usage: client_send_command("127.0.0.1", 6379, "SET foo bar").await
//        client_send_command("127.0.0.1", 6379, "GET foo").await
println!("Client helper defined. Use in async block or tokio::main.");

## 📝 Exercise: Connection timeout and max-connections limit

1. **Connection timeout**: Use `tokio::time::timeout` when accepting or reading from a client.
2. **Max connections**: Track active connections with `Arc<AtomicUsize>`; reject new connections when limit is reached.

In [ ]:
// YOUR CODE HERE
// Add connection timeout and max-connections limit to the server

use std::sync::atomic::{AtomicUsize, Ordering};

// Example: max_connections = Arc::new(AtomicUsize::new(0));
// Before accept: if max_connections.fetch_add(1, Ordering::SeqCst) >= 100 { ... reject ... }
// In handle_client drop: max_connections.fetch_sub(1, Ordering::SeqCst);
// Timeout: tokio::time::timeout(Duration::from_secs(30), stream.read(&mut buf)).await??

println!("Implement connection timeout and max-connections limit.");

## 🎯 Key Takeaways

1. Use `tokio` with `full` features for async TCP server and client
2. Spawn a task per client with `tokio::spawn` for concurrent handling
3. Read commands from `TcpStream`, parse, execute against storage, write response
4. Client: connect, `write_all`, `read` — simple request/response pattern
5. Graceful disconnect: `Ok(0)` from read means client closed; handle `Err` for network drops

---
**Tomorrow:** Feature 2 — Advanced data structures (Lists, Hashes)